In [1]:
import pandas as pd
import json
import ast
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm
from tqdm.auto import tqdm
tqdm.pandas()
import os
import math
import psutil
import gc
import chess
import chess.engine
import chess.svg
from IPython.display import display, HTML
import subprocess
import time
from pathlib import Path
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import matplotlib.patches as patches
import sqlite3
from matplotlib.patches import ConnectionPatch, Rectangle
from functools import reduce

df=pd.read_parquet('skill-elo.parquet')

In [2]:
df.head()

,Player,Week_Idx,Is_Optimal,Δi,Is_Blunder,Is_Error,Cog_Speed,move_count,Time_Management_Score,Weekly_Avg_ELO,ELO_Change
0,4empechement,63,0.500000,77.541832,0.043825,0.087649,0.433976,502,-0.531818,2477.000000,NaN
1,4empechement,70,0.520792,128.621780,0.037624,0.081188,0.458203,505,-0.020909,2401.363636,-27.636364
2,4empechement,71,0.645652,74.480438,0.028261,0.043478,0.461216,460,-0.116364,2373.727273,NaN
3,4empechement,77,0.536585,77.416855,0.031042,0.066519,0.434357,451,-0.389091,2386.000000,NaN
4,4empechement,80,0.567500,113.712502,0.042500,0.070000,0.445507,400,-0.508182,2406.818182,NaN


In [4]:
# 1. 确保数据严格按玩家和时间轴排序
df = df.sort_values(['Player', 'Week_Idx']).copy()

# 2. 直接往下取第 12 次出勤的数据
df['Week_Idx_in_12_Games'] = df.groupby('Player')['Week_Idx'].shift(-12)
df['ELO_in_12_Games'] = df.groupby('Player')['Weekly_Avg_ELO'].shift(-12)

# 3. 【极度严谨的过滤】判断这 12 次出勤是否刚好跨越了 12 周？
# 只有差值严格等于 12 的，才是真正的“全勤无休”
df['Is_Perfect_Attendance'] = (df['Week_Idx_in_12_Games'] - df['Week_Idx']) == 12

# 4. 计算全勤样本的真实 ELO 涨跌
df['Future_12W_Growth'] = np.where(
    df['Is_Perfect_Attendance'], 
    df['ELO_in_12_Games'] - df['Weekly_Avg_ELO'], 
    np.nan
)

# 5. 划定起跑线（绝对不能把 2400 和 2700 放一起比）
bins = [2300, 2400, 2500, 2600, 2700, 2800]
labels = ['2300s', '2400s', '2500s', '2600s', '2700s']
df['ELO_Bracket'] = pd.cut(df['Weekly_Avg_ELO'], bins=bins, labels=labels, right=False)

# 6. 生成最终纯净队列
strict_cohort = df.dropna(subset=['Future_12W_Growth', 'ELO_Bracket', 'Time_Management_Score']).copy()

print(f"大浪淘沙！找到 {len(strict_cohort)} 个 '连续12周全勤' 的纯净成长期样本。")

大浪淘沙！找到 11112 个 '连续12周全勤' 的纯净成长期样本。


In [9]:
strict_cohort.head()

,Player,Week_Idx,Is_Optimal,Δi,Is_Blunder,Is_Error,Cog_Speed,move_count,Time_Management_Score,Weekly_Avg_ELO,ELO_Change,ELO_Bracket,ELO_in_12_Weeks,Week_Idx_in_12_Weeks,Future_ELO_Growth,Week_Idx_in_12_Games,ELO_in_12_Games,Is_Perfect_Attendance,Future_12W_Growth
114,aaditya dhingra,102,0.568548,124.699600,0.038306,0.072581,0.366192,496,0.043636,2458.000000,-47.636364,2400s,2482.181818,114.0,24.181818,114.0,2482.181818,True,24.181818
115,aaditya dhingra,103,0.562929,166.327225,0.052632,0.086957,0.363388,437,-0.329091,2410.363636,19.272727,2400s,2508.636364,115.0,98.272727,115.0,2508.636364,True,98.272727
116,aaditya dhingra,104,0.586420,144.727371,0.044239,0.065844,0.491993,972,0.084091,2429.636364,0.181818,2400s,2531.818182,116.0,102.181818,116.0,2531.818182,True,102.181818
117,aaditya dhingra,105,0.572859,59.761959,0.025584,0.053393,0.437018,899,-0.273810,2429.818182,15.545455,2400s,2519.333333,117.0,89.515152,117.0,2519.333333,True,89.515152
118,aaditya dhingra,106,0.549254,30.731344,0.029851,0.068657,0.633098,335,-0.573000,2445.363636,2.636364,2400s,2511.090909,118.0,65.727273,118.0,2511.090909,True,65.727273


In [11]:
import pandas as pd
from scipy import stats
import numpy as np

# 定义你想要扫描的所有行为特征列
feature_cols = [
    'Is_Optimal', 
    # 'Δi', 
    'Is_Blunder', 'Is_Error', 
    'Cog_Speed', 'Time_Management_Score'
]

print("="*20 + " 全维度行为指纹扫描报告 " + "="*20)

labels = ['2300s', '2400s', '2500s', '2600s', '2700s']

# 创建一个用于存储结果的列表，方便后续转成表格查看
scan_results = []

for bracket in labels:
    bracket_data = strict_cohort[strict_cohort['ELO_Bracket'] == bracket].copy()
    if len(bracket_data) < 100: continue
    
    # 划分 飞升组(Top 20%) 和 平庸组(30%-70%)
    top_threshold = bracket_data['Future_12W_Growth'].quantile(0.80)
    mid_low = bracket_data['Future_12W_Growth'].quantile(0.30)
    mid_high = bracket_data['Future_12W_Growth'].quantile(0.70)

    fast_growers = bracket_data[bracket_data['Future_12W_Growth'] >= top_threshold]
    normal_growers = bracket_data[(bracket_data['Future_12W_Growth'] >= mid_low) & 
                                  (bracket_data['Future_12W_Growth'] <= mid_high)]

    print(f"\n>>> 【{bracket} 段位分析】 (飞升: {len(fast_growers)}人, 平庸: {len(normal_growers)}人)")
    
    for col in feature_cols:
        val_fast = fast_growers[col].dropna()
        val_normal = normal_growers[col].dropna()
        
        t_stat, p_val = stats.ttest_ind(val_fast, val_normal, nan_policy='omit')
        
        # 计算偏离百分比 (飞升组比平庸组强/弱多少)
        mean_diff_pct = ((val_fast.mean() - val_normal.mean()) / abs(val_normal.mean())) * 100
        
        if p_val < 0.05:
            sig = "***" if p_val < 0.01 else "**"
            trend = "↑" if val_fast.mean() > val_normal.mean() else "↓"
            print(f"  [{sig}] {col:25} | 偏离度: {mean_diff_pct:>6.1f}% | 趋势: {trend} (P={p_val:.4f})")
            
            scan_results.append({
                'Bracket': bracket, 'Feature': col, 
                'Diff_Pct': mean_diff_pct, 'P_Value': p_val
            })

# 转换为 DataFrame 方便观察整体规律
summary_df = pd.DataFrame(scan_results)

==================== 全维度行为指纹扫描报告 ====================

>>> 【2300s 段位分析】 (飞升: 344人, 平庸: 687人)
  [***] Cog_Speed                 | 偏离度:   14.8% | 趋势: ↑ (P=0.0000)

>>> 【2400s 段位分析】 (飞升: 564人, 平庸: 1126人)

>>> 【2500s 段位分析】 (飞升: 638人, 平庸: 1275人)

>>> 【2600s 段位分析】 (飞升: 421人, 平庸: 841人)
  [***] Cog_Speed                 | 偏离度:    6.3% | 趋势: ↑ (P=0.0031)
  [***] Time_Management_Score     | 偏离度:  -95.6% | 趋势: ↓ (P=0.0081)

>>> 【2700s 段位分析】 (飞升: 258人, 平庸: 515人)
  [***] Is_Optimal                | 偏离度:   -1.6% | 趋势: ↓ (P=0.0079)
  [***] Time_Management_Score     | 偏离度:  100.7% | 趋势: ↑ (P=0.0010)
